# wandaa / sabyinyo — resumable fine-tune on Kaggle

Mirror of `train_colab.ipynb`, adapted to Kaggle. The whole run is driven by
one launcher, so this notebook and the Colab one are the same three cells —
open either fresh and it **resumes** from the last checkpoint on the Hub.

**Before running:**
1. Notebook settings → **Accelerator: GPU** (P100, or T4 x2).
2. **Add-ons → Secrets**, add these (names matched case-insensitively by the launcher):
   - `HF_TOKEN_WRITE` — a Hugging Face **write** token (checkpoints are pushed).
   - `SABYINYO_ADMIN_TOKEN` — optional; only if you want the unrestricted admin path.
3. Turn **Internet: On** (needed to clone + pull the base model + push to HF).

Kaggle gives ~30h/week of GPU. A run that hits the weekly cap just stops; the
next session resumes from the last pushed checkpoint (every 200 steps / 10 min).


## 1. Clone + install (idempotent)

In [ ]:
import os, subprocess, sys

# Clone once into /kaggle/working (the only writable, persistent-per-session dir).
REPO = "/kaggle/working/sabyinyo"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--branch", "main",
                    "https://github.com/map-boy/sabyinyo.git", REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print("repo:", REPO)

## 2. Launch — clone-aware, resume-aware, platform-aware

`training.launch.launch()` does: install `.[finetune]` (without disturbing
Kaggle's CUDA torch) → read `HF_TOKEN_WRITE` / `SABYINYO_ADMIN_TOKEN` from
**Kaggle Secrets** → check the HF repo for the latest checkpoint → resume or
start fresh → train, checkpointing to `map-boy/sabyinyo-codegen`.

To change base model, LoRA rank, batch size, or checkpoint cadence, edit
`configs/finetune_config.yaml` — every value is saved into each checkpoint so a
resume uses the exact same settings.

In [ ]:
from training.launch import launch
launch(config="configs/finetune_config.yaml", branch="main")

## 3. (Optional) score an intermediate checkpoint

Any checkpoint on the Hub can be evaluated by step number, not just `latest`.
For Path A (from-scratch `CodeGenModel`) checkpoints this scores held-out
perplexity + diagnostics. See `docs/MVP_ARCHITECTURE.md` for the Path A vs
Path C loader split.

In [ ]:
# Requires KAGGLE_USERNAME + KAGGLE_KEY secrets for the corpus, and hug_read.
# !PYTHONPATH=. python eval/run_eval.py --data-dir /kaggle/working/data --checkpoint step_400

## Notes on Kaggle specifics
- **Output:** everything under `/kaggle/working` persists for the session and
  can be saved as a Kaggle dataset; checkpoints also go to HF so they survive
  the session ending.
- **Preemption:** Kaggle sends SIGTERM on stop/timeout — `training/finetune.py`
  catches it and forces one final checkpoint push before exit.
- **Quota:** the ~30h/week GPU cap is not catchable as a signal; the periodic
  safety-net pushes (every 200 steps / 10 min) bound loss if the cap hits mid-run.